# Descargar los archivos para el RAG mediante PubMed

Crear ambiente virtual en CMD

python -m venv env
env\Scripts\activate.bat

In [36]:
%pip install biopython lxml python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
from Bio import Entrez
from lxml import etree
import os
import re
import time
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path("env/.env"))

True

In [41]:
# ===================== CONFIG =====================
Entrez.email = os.getenv("EMAIL")
OUT_DIR = "RAG"
BATCH_SIZE = 50
SLEEP_SEC = 0.34
MAX_FILENAME_LEN = 150
# ==================================================


def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()


def safe_filename_from_title(title: str, fallback: str) -> str:
    base = title if title else fallback
    base = clean_text(base)
    base = re.sub(r'[\\/*?:"<>|]', "", base)
    base = re.sub(r"\s+", "_", base)
    base = re.sub(r"[^a-zA-Z0-9_\.\-]", "", base)
    if not base:
        base = fallback
    return base[:MAX_FILENAME_LEN]


def buscar_pmids_pubmed(termino: str, retmax: int = 50):
    query = f'{termino} AND "free full text"[Filter]'
    handle = Entrez.esearch(db="pubmed", term=query, retmax=retmax)
    rec = Entrez.read(handle)
    handle.close()
    return rec.get("IdList", [])


def pmids_a_pmcids(pmids):
    pmc_ids = []
    for i in range(0, len(pmids), BATCH_SIZE):
        batch = pmids[i:i+BATCH_SIZE]
        handle = Entrez.elink(dbfrom="pubmed", db="pmc", id=",".join(batch))
        rec = Entrez.read(handle)
        handle.close()
        time.sleep(SLEEP_SEC)

        for linkset in rec:
            for ldb in linkset.get("LinkSetDb", []):
                for link in ldb.get("Link", []):
                    if "Id" in link:
                        pmc_ids.append(link["Id"])

    return list(dict.fromkeys(pmc_ids))


def article_to_txt(art):
    pmid_el = art.find(".//front//article-id[@pub-id-type='pmid']")
    pmc_el = art.find(".//front//article-id[@pub-id-type='pmc']")

    pmid = pmid_el.text.strip() if pmid_el is not None and pmid_el.text else ""
    pmc = pmc_el.text.strip() if pmc_el is not None and pmc_el.text else ""

    title_el = art.find(".//front//article-title")
    title = clean_text("".join(title_el.itertext())) if title_el is not None else ""

    abstracts = []
    for abs_el in art.findall(".//front//abstract"):
        abs_txt = clean_text(" ".join(abs_el.itertext()))
        if abs_txt:
            abstracts.append(abs_txt)
    abstract = "\n".join(abstracts)

    body_parts = []
    for sec in art.findall(".//body//sec"):
        sec_title_el = sec.find("./title")
        sec_title = clean_text("".join(sec_title_el.itertext())) if sec_title_el is not None else ""
        sec_txt = clean_text(" ".join(sec.itertext()))

        if sec_title and sec_txt.startswith(sec_title):
            sec_txt = sec_txt[len(sec_title):].strip()

        if sec_txt:
            if sec_title:
                body_parts.append(f"\n## {sec_title}\n{sec_txt}\n")
            else:
                body_parts.append(f"\n{sec_txt}\n")

    body = "\n".join(body_parts).strip()

    doc_id = pmc or pmid or "unknown_id"

    header = [
        f"TITLE: {title}",
        f"PMID: {pmid}",
        f"PMC: {pmc}",
        "-" * 60
    ]

    txt = "\n".join(header) + "\n"

    if abstract:
        txt += "\n# ABSTRACT\n" + abstract + "\n"
    if body:
        txt += "\n# BODY\n" + body + "\n"

    return {
        "doc_id": doc_id,
        "title": title,
        "txt": txt
    }


def descargar_pmc_txt_por_ids(pmc_ids, max_docs, out_dir=OUT_DIR):
    os.makedirs(out_dir, exist_ok=True)
    count = 0

    # Limitamos a max_docs reales
    pmc_ids = pmc_ids[:max_docs]

    for i in range(0, len(pmc_ids), BATCH_SIZE):
        batch = pmc_ids[i:i+BATCH_SIZE]
        handle = Entrez.efetch(db="pmc", id=",".join(batch), rettype="full", retmode="xml")
        xml_bytes = handle.read()
        handle.close()

        root = etree.fromstring(xml_bytes)
        articles = root.findall(".//article")

        for art in articles:
            if count >= max_docs:
                break

            data = article_to_txt(art)

            safe_title = safe_filename_from_title(data["title"], fallback=data["doc_id"])
            safe_id = re.sub(r"[^a-zA-Z0-9_]+", "_", data["doc_id"])
            fname = os.path.join(out_dir, f"{safe_title}_{safe_id}.txt")

            with open(fname, "w", encoding="utf-8") as f:
                f.write(data["txt"])

            count += 1

        time.sleep(SLEEP_SEC)

    return count


def descargar_textos_desde_pubmed(termino: str, max_docs=5):
    # Buscamos más de 5 para asegurarnos de que haya suficientes en PMC
    pmids = buscar_pmids_pubmed(termino, retmax=30)

    if not pmids:
        print("No se hallaron PMIDs.")
        return 0

    pmc_ids = pmids_a_pmcids(pmids)

    if not pmc_ids:
        print("No hay artículos con texto completo en PMC.")
        return 0

    count = descargar_pmc_txt_por_ids(pmc_ids, max_docs=max_docs)
    print(f"Descargados {count} artículos (máximo {max_docs}) para el tema: {termino}")
    return count

In [35]:
temas = ("Differences between DNA replication and RNA transcription", "Prokaryotic vs eukaryotic gene expression regulation", "Comparison of apoptosis, necrosis and autophagy pathways",
         "Active transport vs passive transport in cellular membranes", "Molecular assembly of the pre-initiation complex in transcription", "Mechanism of CRISPR-Cas9 mediated genome editing",
         "Protein folding and chaperone-mediated quality control", "Structure and function of the mitochondrial respiratory chain", "Role of the endoplasmic reticulum in protein synthesis",
         "Structure of the plasma membrane and fluid mosaic model", "Molecular basis of cancer: oncogenes and tumor suppressors", "Modern techniques in molecular biology: PCR and NGS")

for tema in temas: 
    descargar_textos_desde_pubmed(tema, max_docs=5)

Descargados 5 artículos (máximo 5) para el tema: Differences between DNA replication and RNA transcription
Descargados 5 artículos (máximo 5) para el tema: Prokaryotic vs eukaryotic gene expression regulation
Descargados 5 artículos (máximo 5) para el tema: Comparison of apoptosis, necrosis and autophagy pathways
Descargados 5 artículos (máximo 5) para el tema: Active transport vs passive transport in cellular membranes
Descargados 5 artículos (máximo 5) para el tema: Molecular assembly of the pre-initiation complex in transcription
Descargados 5 artículos (máximo 5) para el tema: Mechanism of CRISPR-Cas9 mediated genome editing
Descargados 5 artículos (máximo 5) para el tema: Protein folding and chaperone-mediated quality control
Descargados 5 artículos (máximo 5) para el tema: Structure and function of the mitochondrial respiratory chain
Descargados 5 artículos (máximo 5) para el tema: Role of the endoplasmic reticulum in protein synthesis
Descargados 5 artículos (máximo 5) para el t